# Scheduling, Retries, and Failure Handling in Airflow


## Prerequisites

- Finish **`02_creating_dags.ipynb`**.
- Airflow running: **[`airflow/README.md`](../../airflow/README.md)**.


## Why this matters

In production:

- Tasks fail
- Networks break
- APIs time out

Good pipelines:

- **Retry** automatically
- **Log** failures
- **Recover** where possible

👉 Reliability is an explicit design choice — not luck.


## Scheduling in Airflow

Define **when** a DAG runs with **`schedule_interval`** (Airflow 2.x still supports this name widely in examples):

- **`@daily`**, **`@hourly`** — presets
- **Cron string** — full control

Examples:

- **`"0 2 * * *"`** → 02:00 every day

👉 Match schedules to **SLAs** and **upstream data availability**.


## Example DAG with a cron schedule

Repo file: **`../../airflow/dags/scheduled_dag.py`** (`dag_id`: **`scheduled_dag`**).

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def task():
    print("Running scheduled task")

with DAG(
    dag_id="scheduled_dag",
    start_date=datetime(2024, 1, 1),
    schedule_interval="0 2 * * *",
    catchup=False,
) as dag:

    t1 = PythonOperator(
        task_id="task1",
        python_callable=task,
    )
```

**`catchup=False`** avoids backfilling many historical runs when you first enable the DAG.


## Retry mechanism (`default_args`)

Put **`retries`** and **`retry_delay`** on **`default_args`** so every task inherits sensible behavior.

Repo file: **`../../airflow/dags/retry_dag.py`** (`dag_id`: **`retry_dag`**).

```python
from datetime import timedelta

with DAG(
    dag_id="retry_dag",
    start_date=datetime(2024, 1, 1),
    schedule_interval="@daily",
    catchup=False,
    default_args={
        "retries": 3,
        "retry_delay": timedelta(seconds=10),
    },
) as dag:

    def unstable_task():
        import random
        if random.random() < 0.7:
            raise Exception("Random failure")
        print("Success")

    t1 = PythonOperator(
        task_id="unstable_task",
        python_callable=unstable_task,
    )
```

👉 Airflow **re-schedules** the task after the delay, up to **`retries`**. Watch **task tries** in the UI.


## Handling failures in code

You can catch exceptions to **log context**, but if the task should fail for Airflow, **re-raise** so the task state stays **`failed`** until retries succeed or exhaust.

```python
def safe_task():
    try:
        print("Processing...")
        raise RuntimeError("Error occurred")
    except Exception as e:
        print("Handled error:", e)
        raise
```

👉 Use **`logging`** (and structured logs in real systems) — don’t swallow failures silently.


## Alerts (concept)

Airflow can:

- Email on failure (SMTP / `email_on_failure`)
- Integrate with Slack, PagerDuty, etc. via callbacks or external observers

👉 Production stacks usually centralize **alerting** with your observability platform.


## Timeouts (`execution_timeout`)

Prevent tasks from **running forever** (wedged jobs, hanging HTTP calls).

Repo file: **`../../airflow/dags/timeout_dag.py`** (`dag_id`: **`timeout_dag`**).

```python
with DAG(
    dag_id="timeout_dag",
    start_date=datetime(2024, 1, 1),
    schedule_interval="@daily",
    catchup=False,
    default_args={
        "execution_timeout": timedelta(seconds=30),
    },
) as dag:
    ...
```

👉 Combine timeout with **retries** only when a **short** hung attempt might succeed on retry.


## Production DAG behavior (checklist)

- **Scheduled** automatically
- **Retries** on transient failures
- **Logs** for every attempt
- **Alerts** when something truly breaks
- **Timeouts** so work doesn’t stall the queue

👉 Automation + guardrails = operable pipelines.


## Practice

1. DAG with **`retries = 2`** and **`retry_delay = 5`** seconds.
2. Task that **randomly fails** (like **`retry_dag`**).
3. Trigger runs and **watch retries** in Grid / task instance details.


## Assignment

**Upgrade an ETL-style DAG** with:

- Scheduling (`schedule_interval`)
- Retry logic (`default_args`)
- **`execution_timeout`**
- **Simulate failure** and observe retries

**Reference implementation:** **`../../airflow/dags/etl_resilient_dag.py`** (`dag_id`: **`etl_resilient_dag`**). Set **`SIMULATE_TRANSFORM_FAILURE = True`** at the top of that file to simulate failures and watch retries.

**Bonus:** append logs with **`logging.FileHandler`** or rely on Airflow’s captured task logs — both are valid patterns.


## Interview questions

1. How does Airflow handle failures?
2. What is retry logic?
3. What is scheduling in Airflow?
4. How do you handle long-running tasks?


## What you just learned

- Scheduling pipelines
- Retry mechanisms
- Failure handling and timeouts
- Production reliability mindset

👉 This is what teams expect from data engineers operating batch systems.


---

**Next:** **Integrating your ETL pipeline with Airflow** — wire real Python ETL into scheduled DAGs.


**Reality check:** a DAG file is not “done” until **schedule**, **retries**, **timeouts**, and **observability** match how the job behaves in production.


## Your tasks

- [ ] Read **`scheduled_dag.py`**, **`retry_dag.py`**, **`timeout_dag.py`** under **`../../airflow/dags/`**.
- [ ] Enable **`retry_dag`**, trigger it multiple times, **observe retries** for **`unstable_task`**.
- [ ] Open **`etl_resilient_dag`**, set **`SIMULATE_TRANSFORM_FAILURE = True`**, trigger and watch **retry + timeout** settings.
- [ ] Optional: answer interview questions in a few sentences.
